# Linear Regression — Mathematics

**Goal.** Build the linear-regression / OLS theory from the ground up: write the model in matrix form, derive the MSE loss from probabilistic assumptions, compute its gradient and Hessian, prove the loss is convex, state existence and uniqueness, derive the closed-form solution, interpret it geometrically via the hat matrix, and handle the singular case via the pseudoinverse.

**Role of this notebook.** Pure mathematics — symbols, definitions, derivations, theorems. The intuition belongs in `01_intuition.ipynb` (pictures, sliders, residuals). The implementation belongs in `05_hands_on_programming.ipynb`. This notebook stays text-only on purpose.

**Prerequisites.** `01_intuition.ipynb` — the visual picture of data, lines, residuals, and the bowl-shaped loss. Comfort with multivariable calculus and basic linear algebra (matrix transpose, symmetric matrices, Gram matrices, eigenvalues, orthogonal projection, the SVD).

**Stage map.** `01_intuition` → **`02_mathematics`** → `03_optimization` → `04_statistics` → `05_hands_on_programming`.

**Theorem stack.**

1. **Model.** How do we write the linear model in matrix form?
2. **Loss.** Why MSE? (Negative log-likelihood under Gaussian noise.)
3. **Gradient and Hessian.** Where does $\nabla L$ point, and why is L convex?
4. **Optimum.** What characterises the minimum (normal equations), and when is it unique?
5. **Geometric meaning.** What is $\theta^*$ as a linear-algebra object? (Projection of y onto $\text{Col}(X)$ via the hat matrix.)
6. **Singular case.** What replaces $(X^\top X)^{-1}$ when $X^\top X$ is singular? (Moore–Penrose pseudoinverse via SVD.)

---

**Reading conventions.** Single-line equations that name an object appear in a blockquote so they stand out from prose. Multi-line derivations with aligned $=$ appear in display math. Theorem statements appear in blockquotes; proofs are signed off with ∎. Equations are numbered, e.g. (3.2), only when later sections refer back to them.

## 0. Notation

Every symbol used later, defined once.

| Symbol | Type | Meaning |
|---|---|---|
| $n$ | scalar $\in \mathbb{N}$ | number of training examples |
| $p$ | scalar $\in \mathbb{N}$ | number of features (including the intercept after the bias trick) |
| $x_i$ | vector $\in \mathbb{R}^p$ | feature vector of the $i$-th example |
| $y_i$ | scalar $\in \mathbb{R}$ | target of the $i$-th example |
| $X$ | matrix $\in \mathbb{R}^{n \times p}$ | design matrix; row $i$ is $x_i^\top$ |
| $y$ | vector $\in \mathbb{R}^n$ | target vector with entries $y_i$ |
| $\theta$ | vector $\in \mathbb{R}^p$ | model parameters (weights) |
| $\hat{y}_i$ | scalar | model prediction for example $i$; $\hat{y}_i = x_i^\top \theta$ |
| $\hat{y}$ | vector $\in \mathbb{R}^n$ | prediction vector; $\hat{y} = X \theta$ |
| $r_i$ | scalar | residual for example $i$; $r_i = \hat{y}_i - y_i$ |
| $r^*$ | vector $\in \mathbb{R}^n$ | residual vector at the optimum; $r^* = \hat{y}^* - y$ |
| $L(\theta)$ | scalar function | the MSE loss; $L : \mathbb{R}^p \to \mathbb{R}^+$ |
| $\nabla L(\theta)$ | vector $\in \mathbb{R}^p$ | gradient of $L$ at $\theta$ |
| $\nabla^2 L(\theta)$ | matrix $\in \mathbb{R}^{p \times p}$ | Hessian of $L$ at $\theta$ |
| $\theta^*$ | vector $\in \mathbb{R}^p$ | a minimiser of $L$ |
| $H$ | matrix $\in \mathbb{R}^{n \times n}$ | hat matrix; $H := X (X^\top X)^{-1} X^\top$ |
| $X^+$ | matrix $\in \mathbb{R}^{p \times n}$ | Moore–Penrose pseudoinverse of $X$ |
| $\varepsilon_i$ | scalar (random) | noise term in the probabilistic model |
| $\sigma^2$ | scalar $> 0$ | noise variance |
| $\|\cdot\|$ | scalar | Euclidean ($\ell^2$) norm |
| $\langle \cdot, \cdot \rangle$ | scalar | Euclidean inner product; $\langle a, b \rangle = a^\top b$ |
| $\text{Col}(X)$ | subspace $\subseteq \mathbb{R}^n$ | column space of $X$ (span of its columns) |
| $\text{Null}(X)$ | subspace $\subseteq \mathbb{R}^p$ | null space of $X$; $\{ v : Xv = 0 \}$ |
| $\text{rank}(X)$ | scalar $\in \mathbb{N}$ | $\dim \text{Col}(X)$; for $X \in \mathbb{R}^{n \times p}$, $\text{rank}(X) \le \min(n, p)$ |

**Conventions used throughout.**

- All vectors are *column* vectors; lowercase letters denote vectors, uppercase letters matrices.
- $A^\top$ is the transpose of $A$; $A^{-1}$ its inverse (when it exists); $A^+$ its Moore–Penrose pseudoinverse.
- Subscripts index examples ($x_i$); commas separate row/column indices when both are needed ($X_{i,j}$).
- A matrix is **symmetric** if $A^\top = A$; **positive semi-definite (PSD)** if $v^\top A v \ge 0$ for all $v$; **positive definite (PD)** if $v^\top A v > 0$ for all $v \neq 0$.
- An $n \times n$ matrix $P$ is an **orthogonal projection** iff $P^\top = P$ and $P^2 = P$.
- Multiplication: a centred dot $\cdot$ separates *scalar* factors (e.g. $(1/n) \cdot L$); juxtaposition is reserved for matrix–vector and matrix–matrix products (e.g. $X \theta$, $X^\top X$).

## 1. Model: linear regression in matrix form

### 1.1 Definition (linear model)

A **linear model** assumes the prediction is a linear combination of features. For each example $i \in \{1, \dots, n\}$ and an unknown parameter vector $\theta \in \mathbb{R}^p$:

> $$y_i = x_{i1} \theta_1 + x_{i2} \theta_2 + \dots + x_{ip} \theta_p = \langle x_i, \theta \rangle = x_i^\top \theta$$

### 1.2 Matrix form

Stack the per-example equations row by row. Define the **design matrix** $X \in \mathbb{R}^{n \times p}$ as the matrix whose $i$-th row equals $x_i^\top$, the **target vector** $y \in \mathbb{R}^n$ as the column vector with entries $y_i$, and the **prediction vector** $\hat{y} \in \mathbb{R}^n$ as the column vector with entries $\hat{y}_i$. Then the $n$ scalar equations $\hat{y}_i = x_i^\top \theta$ collapse into a single matrix–vector product:

> $$\hat{y} = X\theta \in \mathbb{R}^n \tag{1.1}$$

### 1.3 Bias trick

A real model has an intercept (bias) term $\theta_0$, so that

> $$\hat{y}_i = \theta_0 + x_{i1} \theta_1 + \dots + x_{ip} \theta_p$$

Absorb $\theta_0$ into $\theta$ by prepending a column of 1s to the feature matrix. Let $\mathbb{1} := (1, 1, \dots, 1)^\top \in \mathbb{R}^n$. Then

> $$\tilde{X} := [ \mathbb{1} \mid X ] \in \mathbb{R}^{n \times (p+1)}, \quad \tilde{\theta} := (\theta_0, \theta_1, \dots, \theta_p)^\top \in \mathbb{R}^{p+1}, \quad \hat{y} = \tilde{X} \tilde{\theta}$$

From here on we drop the tildes and assume $X$ already includes the bias column; $p$ denotes the total number of weights (including $\theta_0$).

## 2. Loss: mean squared error

### 2.1 Definition (residual and MSE)

The **residual** for example $i$ is the signed error

> $$r_i(\theta) := \hat{y}_i - y_i = x_i^\top \theta - y_i$$

The **mean squared error** is the averaged squared residual. Written equivalently in scalar, vector, and quadratic-expansion forms:

$$\begin{aligned}
L(\theta) &= \frac{1}{n} \sum_i r_i(\theta)^2 & \text{(scalar form)} \\
&= \frac{1}{n} \cdot \|X\theta - y\|^2 & \text{(vector form)} & \quad (2.1) \\
&= \frac{1}{n} \cdot \bigl(\theta^\top X^\top X \theta - 2\, y^\top X \theta + y^\top y\bigr) & \text{(expanded quadratic)} & \quad (2.2)
\end{aligned}$$

**Ordinary least squares (OLS)** is the optimisation problem

> $$\theta^* \in \arg\min_{\theta \in \mathbb{R}^p} L(\theta)$$

### 2.2 Theorem (MSE is the Gaussian negative log-likelihood)

Squaring is not an aesthetic choice — it is forced by a probabilistic model.

> **Theorem 2.2.** Assume the data-generating process
>
> $$y_i = x_i^\top \theta + \varepsilon_i, \quad \varepsilon_i \sim \mathcal{N}(0, \sigma^2) \text{ i.i.d.}, \quad i = 1, \ldots, n.$$
>
> Then the maximum-likelihood estimator of $\theta$ given $(X, y)$ is identical to the OLS minimiser of $L(\theta)$.

**Proof.** Conditional on $X$, the residuals $\varepsilon_i = y_i - x_i^\top \theta$ are i.i.d. $\mathcal{N}(0, \sigma^2)$, so the joint density factorises:

$$p(y \mid X, \theta) = \prod_i (2\pi \sigma^2)^{-1/2} \cdot \exp\!\left( -\frac{(y_i - x_i^\top \theta)^2}{2\sigma^2} \right).$$

Take logarithms and use the definition of $L(\theta)$:

$$\begin{aligned}
\ell(\theta) &:= \log p(y \mid X, \theta) \\
&= -\frac{n}{2} \cdot \log(2\pi \sigma^2) - \frac{1}{2\sigma^2} \cdot \sum_i (y_i - x_i^\top \theta)^2 \\
&= -\frac{n}{2} \cdot \log(2\pi \sigma^2) - \frac{n}{2\sigma^2} \cdot L(\theta).
\end{aligned}$$

The first term does not depend on $\theta$, and $n / (2\sigma^2) > 0$ is a positive constant. Therefore

> $$\arg\max_\theta \ell(\theta) = \arg\min_\theta L(\theta),$$

which is exactly the OLS minimiser. ∎

**Remarks.**

- Different noise distributions give different losses: $\varepsilon_i \sim \text{Laplace}(0, b)$ yields mean absolute error (MAE); a Gaussian–Laplacian mixture yields the Huber loss.
- Differentiability of $L$ is automatic (each square is $C^\infty$), and a single residual of magnitude 10 contributes the same to $L$ as 100 residuals of magnitude 1 — a quadratic penalty for outliers, baked in for free.

## 3. Gradient and Hessian

### 3.1 Matrix calculus identities

Two identities power all of OLS calculus. Both are stated for column vectors $\theta \in \mathbb{R}^p$.

**Identity (i) — linear form.** For a constant vector $b \in \mathbb{R}^p$,

> $$\nabla_\theta (b^\top \theta) = b.$$

*Proof.* $b^\top \theta = \sum_j b_j \theta_j$, hence $\partial(b^\top \theta)/\partial\theta_k = b_k$ for each $k$. Stacking gives the column vector $b$. ∎

**Identity (ii) — quadratic form.** For a constant matrix $A \in \mathbb{R}^{p \times p}$,

> $$\nabla_\theta (\theta^\top A \theta) = (A + A^\top) \theta.$$

*Proof.* Write $\theta^\top A \theta = \sum_{jk} A_{jk} \theta_j \theta_k$. Differentiate with respect to $\theta_l$:

$$\frac{\partial}{\partial\theta_l} \left( \sum_{jk} A_{jk} \theta_j \theta_k \right) = \sum_k A_{lk} \theta_k + \sum_j A_{jl} \theta_j = (A\theta)_l + (A^\top \theta)_l.$$

Stacking over $l$ gives $(A + A^\top) \theta$. ∎

*Special case.* When $A$ is symmetric ($A = A^\top$), the identity reduces to $\nabla_\theta (\theta^\top A \theta) = 2 A \theta$.

### 3.2 Theorem (gradient of L)

> **Theorem 3.2.**   $\nabla L(\theta) = \frac{2}{n} \cdot X^\top (X \theta - y)$.   (3.1)

**Proof.** Start from the expanded form (2.2):

$$L(\theta) = \frac{1}{n} \cdot \bigl( \theta^\top X^\top X \theta - 2\, y^\top X \theta + y^\top y \bigr).$$

Differentiate term by term.

- *Quadratic term.* Apply Identity (ii) with $A = X^\top X$. The matrix $X^\top X$ is symmetric ($(X^\top X)^\top = X^\top X$), so its gradient is $2 X^\top X \theta$.
- *Linear term.* Rewrite $-2\, y^\top X \theta = -2 (X^\top y)^\top \theta$ and apply Identity (i) with $b = X^\top y$, yielding $-2 X^\top y$.
- *Constant term.* $y^\top y$ does not depend on $\theta$, so its gradient is $0$.

Combining:

$$\nabla L(\theta) = \frac{1}{n} \cdot \bigl( 2 X^\top X \theta - 2 X^\top y \bigr) = \frac{2}{n} \cdot X^\top (X \theta - y). \quad \blacksquare$$

### 3.3 Theorem (Hessian of L)

> **Theorem 3.3.**   $\nabla^2 L(\theta) = \frac{2}{n} \cdot X^\top X$,    independent of $\theta$.

**Proof.** From (3.1), $\nabla L(\theta) = \frac{2}{n} \cdot (X^\top X \theta - X^\top y)$. The map $\theta \mapsto \frac{2}{n} \cdot X^\top X \theta$ is linear in $\theta$ with constant Jacobian $\frac{2}{n} \cdot X^\top X$; the constant term $-\frac{2}{n} \cdot X^\top y$ has zero Jacobian. ∎

### 3.4 Theorem (convexity of L)

> **Theorem 3.4.**
> 
> 1. $L$ is convex on $\mathbb{R}^p$.
> 2. $L$ is **strictly** convex if and only if $\text{rank}(X) = p$.

**Proof.** Standard fact (multivariable calculus): a twice-differentiable function on $\mathbb{R}^p$ is convex iff its Hessian is PSD everywhere, and strictly convex iff its Hessian is PD everywhere. By Theorem 3.3, $\nabla^2 L = \frac{2}{n} \cdot X^\top X$, so it suffices to study $X^\top X$.

*(1) $X^\top X$ is PSD.* For any $v \in \mathbb{R}^p$,

$$v^\top (X^\top X) v = (Xv)^\top (Xv) = \|Xv\|^2 \ge 0.$$

Hence $X^\top X$ is PSD, $\nabla^2 L$ is PSD, and $L$ is convex.

*(2) PD ⟺ full column rank.* From the same identity, $X^\top X$ is PD iff $\|Xv\|^2 > 0$ for every $v \neq 0$, iff $Xv = 0$ implies $v = 0$, iff $\text{Null}(X) = \{0\}$, iff the columns of $X$ are linearly independent, iff $\text{rank}(X) = p$. ∎

**Reading.** The "bowl" of `01_intuition.ipynb` §3 is now a theorem: $L$ is a convex quadratic, *strictly* convex whenever the features are linearly independent. Strict convexity forces a unique minimiser; mere convexity only guarantees a convex *set* of minimisers.

## 4. Optimum: the normal equations

### 4.1 First-order optimality

Since $L$ is convex (Theorem 3.4 part 1), every critical point of $L$ is a *global* minimiser. So we set $\nabla L(\theta) = 0$ in (3.1) and cancel the constant $2/n$:

> $$X^\top X \theta = X^\top y \tag{4.1}$$

Equation (4.1) is the system of **normal equations** for OLS.

### 4.2 Theorem (existence and uniqueness)

> **Theorem 4.2.** Let $X \in \mathbb{R}^{n \times p}$, $y \in \mathbb{R}^n$, and $L(\theta) = \frac{1}{n} \cdot \|X\theta - y\|^2$. Define $\Theta^* := \arg\min_\theta L(\theta)$. Then
> 
> 1. **(Existence.)** $\Theta^*$ is non-empty.
> 2. **(Uniqueness.)** $|\Theta^*| = 1$ $\iff$ $\text{rank}(X) = p$ $\iff$ $X^\top X$ is invertible.
> 3. **(Closed form.)** When $\text{rank}(X) = p$, the unique minimiser is
> 
>     $$\theta^* = (X^\top X)^{-1} X^\top y. \tag{4.2}$$
> 
> When $\text{rank}(X) < p$, $\Theta^*$ is an affine subspace of $\mathbb{R}^p$ of dimension $p - \text{rank}(X)$. §6 selects its unique minimum-norm element via the pseudoinverse.

**Proof.**

*(1) Existence.* A vector $\theta$ minimises $L$ iff it satisfies the normal equations (4.1). The system is consistent because $X^\top y$ belongs to the column space of $X^\top X$: indeed $\text{Col}(X^\top X) = \text{Col}(X^\top)$ (a consequence of $\text{Null}(X) = \text{Null}(X^\top X)$ and rank–nullity), and $X^\top y \in \text{Col}(X^\top)$ trivially. Hence $\Theta^* \neq \varnothing$.

*(2) Uniqueness.* By Theorem 3.4 part 2, $L$ is strictly convex iff $\text{rank}(X) = p$. A strictly convex function has at most one minimiser; combined with (1), it has exactly one. Conversely, suppose $\text{rank}(X) < p$ and pick $\theta^* \in \Theta^*$ together with any $v \in \text{Null}(X) \setminus \{0\}$. Then $Xv = 0$, so for every $\alpha \in \mathbb{R}$,

$$L(\theta^* + \alpha v) = \frac{1}{n} \cdot \|X(\theta^* + \alpha v) - y\|^2 = \frac{1}{n} \cdot \|X \theta^* - y\|^2 = L(\theta^*).$$

Hence $\theta^* + \alpha v \in \Theta^*$ for every $\alpha$, giving $|\Theta^*| = \infty$.

*(3) Closed form.* Under $\text{rank}(X) = p$, $X^\top X$ is invertible (Theorem 3.4), so (4.1) has the unique solution $\theta^* = (X^\top X)^{-1} X^\top y$. ∎

### 4.3 Computational note

Although $(X^\top X)^{-1} X^\top y$ is the cleanest *mathematical* expression for $\theta^*$, it is rarely the best *numerical* path. Forming $X^\top X$ squares the condition number of $X$, so direct solvers based on a QR or SVD factorisation of $X$ (without forming $X^\top X$) are preferred in practice. The closed form also costs $\Theta(n p^2 + p^3)$ time and $\Theta(p^2)$ memory, which is impractical for large $p$ — this is what motivates `03_optimization.ipynb`.

## 5. Geometric meaning: hat matrix and orthogonal projection

### 5.1 Residual orthogonality

Rewrite the normal equations (4.1) as

> $$X^\top (X \theta^* - y) = 0$$    i.e.    $X^\top r^* = 0$,    where $r^* := X \theta^* - y$.   (5.1)

Equation (5.1) says $r^*$ is orthogonal to every column of $X$, hence to every vector in $\text{Col}(X)$. This is the linear-algebra signature of an **orthogonal projection**: $\hat{y}^* = X \theta^*$ is the unique vector in $\text{Col}(X)$ such that $y - \hat{y}^* \perp \text{Col}(X)$.

### 5.2 Definition (hat matrix)

Assume $\text{rank}(X) = p$, so $(X^\top X)^{-1}$ exists. Substituting $\theta^* = (X^\top X)^{-1} X^\top y$ into $\hat{y}^* = X \theta^*$:

> $$\hat{y}^* = X (X^\top X)^{-1} X^\top y = H y$$    where    $H := X (X^\top X)^{-1} X^\top \in \mathbb{R}^{n \times n}$.   (5.2)

$H$ is called the **hat matrix** — it puts the hat on $y$.

### 5.3 Theorem ($H$ is the orthogonal projection onto $\text{Col}(X)$)

> **Theorem 5.3.** The hat matrix $H$ satisfies
> 
> 1. **Symmetry.**   $H^\top = H$.
> 2. **Idempotence.**   $H^2 = H$.
> 3. **Range.**   $\text{Im}(H) = \text{Col}(X)$.
> 4. **Spectrum.**   Every eigenvalue of $H$ is $0$ or $1$, and $\text{rank}(H) = \text{trace}(H) = p$.
> 
> Properties (1)–(2) together characterise an orthogonal projection matrix in $\mathbb{R}^n$.

**Proof.**

*(1) Symmetry.* Use $(AB)^\top = B^\top A^\top$ and the symmetry of $X^\top X$ (so $((X^\top X)^{-1})^\top = (X^\top X)^{-1}$):

$$H^\top = \bigl( X (X^\top X)^{-1} X^\top \bigr)^\top = X \bigl( (X^\top X)^{-1} \bigr)^\top X^\top = X (X^\top X)^{-1} X^\top = H.$$

*(2) Idempotence.* Expand $H^2$, cancelling the central $(X^\top X)(X^\top X)^{-1} = I_p$:

$$H^2 = X (X^\top X)^{-1} X^\top X (X^\top X)^{-1} X^\top = X (X^\top X)^{-1} X^\top = H.$$

*(3) Range.* For any $y$, $Hy = X \cdot [(X^\top X)^{-1} X^\top y] \in \text{Col}(X)$, so $\text{Im}(H) \subseteq \text{Col}(X)$. Conversely, take $z \in \text{Col}(X)$; write $z = Xw$ for some $w \in \mathbb{R}^p$. Then

$$Hz = X (X^\top X)^{-1} X^\top X w = X w = z,$$

so $\text{Col}(X) \subseteq \text{Im}(H)$. Combining, $\text{Im}(H) = \text{Col}(X)$.

*(4) Spectrum.* If $Hv = \lambda v$ with $v \neq 0$, then $H^2 v = \lambda^2 v$; using $H^2 = H$,

$$\lambda^2 v = \lambda v \implies \lambda(\lambda - 1)v = 0 \implies \lambda \in \{0, 1\}.$$

The multiplicity of the eigenvalue $1$ equals $\dim \text{Im}(H) = \dim \text{Col}(X) = \text{rank}(X) = p$. The trace of $H$ equals the sum of its eigenvalues $= p$, which also equals $\text{rank}(H)$. ∎

### 5.4 Corollary (Pythagoras)

> **Corollary 5.4.**
>
> $$\|y\|^2 = \|\hat{y}^*\|^2 + \|r^*\|^2 \tag{5.3}$$

**Proof.** From (5.1), $r^* \perp \text{Col}(X)$; from (5.2), $\hat{y}^* \in \text{Col}(X)$. Therefore $\langle \hat{y}^*, r^* \rangle = 0$. Now decompose $y = \hat{y}^* + r^*$ (since $r^* = \hat{y}^* - y$ means $y = \hat{y}^* - r^*$):

$$\|y\|^2 = \|\hat{y}^* - r^*\|^2 = \|\hat{y}^*\|^2 - 2\langle \hat{y}^*, r^* \rangle + \|r^*\|^2 = \|\hat{y}^*\|^2 + \|r^*\|^2. \quad \blacksquare$$

**Reading.** OLS is not just "the line that minimises a sum". It is *the* orthogonal projection of $y$ onto the subspace $\text{Col}(X)$ of achievable predictions. Pythagoras decomposes the total squared length of $y$ into an **explained** part $\|\hat{y}^*\|^2$ and an **unexplained** part $\|r^*\|^2$ — the seed of the $R^2$ coefficient developed in `04_statistics.ipynb`.

## 6. Singular case: SVD and the Moore–Penrose pseudoinverse

When $\text{rank}(X) < p$ (the *multicollinear* regime), $X^\top X$ is singular, the closed form (4.2) is undefined, and $\Theta^*$ is an infinite affine subspace by Theorem 4.2 part 2. The singular value decomposition lets us still write down a canonical minimiser.

### 6.1 The singular value decomposition

> **Theorem 6.1 (SVD).** Every $X \in \mathbb{R}^{n \times p}$ admits a factorisation
> 
> $$X = U \Sigma V^\top$$
> 
> where $U \in \mathbb{R}^{n \times n}$ and $V \in \mathbb{R}^{p \times p}$ are orthogonal ($U^\top U = I_n$, $V^\top V = I_p$) and $\Sigma \in \mathbb{R}^{n \times p}$ is diagonal with non-negative entries
> 
> $$\sigma_1 \ge \sigma_2 \ge \cdots \ge \sigma_{\min(n,p)} \ge 0.$$
> 
> The non-zero $\sigma_i$ are the **singular values** of $X$; exactly $r := \text{rank}(X)$ of them are positive.

*(Statement only; see e.g. Trefethen–Bau §4–5 for a constructive proof.)*

### 6.2 Definition (Moore–Penrose pseudoinverse)

Given the SVD $X = U \Sigma V^\top$, build $\Sigma^+ \in \mathbb{R}^{p \times n}$ by inverting the *non-zero* singular values:

$$\begin{aligned}
(\Sigma^+)_{ii} &= 1 / \sigma_i & \text{if } \sigma_i > 0, \\
(\Sigma^+)_{ii} &= 0 & \text{if } \sigma_i = 0, \\
(\Sigma^+)_{ij} &= 0 & \text{for } i \neq j.
\end{aligned}$$

The **Moore–Penrose pseudoinverse** of $X$ is

> $$X^+ := V \Sigma^+ U^\top \in \mathbb{R}^{p \times n} \tag{6.1}$$

$X^+$ exists and is unique for every $X$ (no rank assumption required).

### 6.3 Theorem ($X^+$ extends the inverse and selects the minimum-norm minimiser)

> **Theorem 6.3.**
> 
> 1. If $\text{rank}(X) = p$, then $X^+ = (X^\top X)^{-1} X^\top$. Hence $\theta^* = X^+ y$ agrees with the OLS closed form (4.2).
> 2. For arbitrary $X$, the vector $\theta_{\text{minnorm}} := X^+ y$ is the **unique** element of $\Theta^*$ with smallest Euclidean norm:
> 
>     $$\theta_{\text{minnorm}} = \arg\min \{ \|\theta\| : \theta \in \Theta^* \}$$
> 
> 3. The prediction $X \cdot X^+ y$ is the orthogonal projection of $y$ onto $\text{Col}(X)$, regardless of rank.

**Proof sketch.**

*(1)* When $\text{rank}(X) = p$, all $p$ singular values are positive and $\Sigma^+ \Sigma = I_p$. Compute:

$$\begin{aligned}
(X^\top X)^{-1} X^\top &= (V \Sigma^\top U^\top U \Sigma V^\top)^{-1} V \Sigma^\top U^\top \\
&= V (\Sigma^\top \Sigma)^{-1} V^\top V \Sigma^\top U^\top \\
&= V (\Sigma^\top \Sigma)^{-1} \Sigma^\top U^\top \\
&= V \Sigma^+ U^\top \\
&= X^+.
\end{aligned}$$

*(2)* Any $\theta \in \Theta^*$ differs from $\theta_{\text{minnorm}}$ by an element of $\text{Null}(X)$ (since $\Theta^*$ is the affine flat $\theta_{\text{minnorm}} + \text{Null}(X)$). The SVD shows $X^+ y \in \text{Col}(X^\top) = \text{Null}(X)^\perp$, so $\theta_{\text{minnorm}} \perp \text{Null}(X)$. By Pythagoras in $\mathbb{R}^p$, for every $v \in \text{Null}(X)$,

$$\|\theta_{\text{minnorm}} + v\|^2 = \|\theta_{\text{minnorm}}\|^2 + \|v\|^2 \ge \|\theta_{\text{minnorm}}\|^2,$$

with equality iff $v = 0$.

*(3)* From the SVD, $X X^+ = U \Sigma \Sigma^+ U^\top$ projects onto the first $r$ columns of $U$, which span $\text{Col}(X)$. ∎

### 6.4 Reading

When the design is multicollinear, the *parameters* $\theta$ are non-unique, but the *predictions* $\hat{y}$ are. $\text{Col}(X)$ is unchanged by collinearity (duplicating a column does not enlarge the span), so the projection of $y$ onto $\text{Col}(X)$ is unchanged; only the coordinates *inside* $\text{Col}(X)$ become ambiguous. The pseudoinverse picks the shortest coordinate vector that lands on the projection.

This is what numerical libraries (e.g. NumPy's `lstsq` based on LAPACK's `gelsd`) compute under the hood: SVD-based, no inversion of $X^\top X$, no assumption of full column rank, and the minimum-norm solution as a free tiebreak.

## Takeaway

- **Model.**   $\hat{y} = X \theta$, with the intercept absorbed by the bias trick $\tilde{X} = [\mathbb{1} \mid X]$.
- **Loss.**   $L(\theta) = \frac{1}{n} \cdot \|X\theta - y\|^2$ is the negative log-likelihood under i.i.d. Gaussian noise (Theorem 2.2); OLS = MLE.
- **Gradient.**   $\nabla L(\theta) = \frac{2}{n} \cdot X^\top (X \theta - y)$   (Theorem 3.2).
- **Hessian.**   $\nabla^2 L(\theta) = \frac{2}{n} \cdot X^\top X$ — constant, PSD; PD iff $\text{rank}(X) = p$   (Theorem 3.3 + 3.4). Hence $L$ is convex; strictly convex iff $X$ has full column rank.
- **Optimum.**   $\Theta^*$ is non-empty; $|\Theta^*| = 1$ iff $\text{rank}(X) = p$; in that case $\theta^* = (X^\top X)^{-1} X^\top y$   (Theorem 4.2).
- **Geometry.**   $\hat{y}^* = Hy$ with $H := X (X^\top X)^{-1} X^\top$ symmetric, idempotent, of rank $p$ — the orthogonal projection onto $\text{Col}(X)$ (Theorem 5.3). Pythagoras: $\|y\|^2 = \|\hat{y}^*\|^2 + \|r^*\|^2$ (Corollary 5.4).
- **Singular case.**   $\theta_{\text{minnorm}} = X^+ y$ from the SVD $X = U \Sigma V^\top$ is the minimum-norm element of $\Theta^*$; it agrees with (4.2) whenever $\text{rank}(X) = p$   (Theorem 6.3).

Next: `03_optimization.ipynb` — when the closed form is too expensive (large $p$) or unavailable (non-MSE loss), gradient descent walks down the convex bowl proven here. The formula $\nabla L(\theta) = \frac{2}{n} \cdot X^\top (X \theta - y)$ derived in Theorem 3.2 is the one it uses.